In [1]:
import json
import re
import pandas as pd
from datetime import datetime, timezone
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

In [2]:
# ============================================================
# CONFIG
# ============================================================

PLAYER_URL = "https://puckpedia.com/player/leo-carlsson"

In [3]:
# ============================================================
# HELPERS
# ============================================================

def clean_text(value):
    if value is None: return None
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value or None

def parse_money(value):
    if not value: return None
    text = str(value).replace("$", "").replace(",", "").strip().upper()
    match = re.search(r"([\d.]+)\s*([KMB])?", text)
    if not match: return None
    number = float(match.group(1))
    multiplier = {"K":1_000,"M":1_000_000,"B":1_000_000_000}.get(match.group(2),1)
    return int(number * multiplier)

def parse_int(value):
    if value is None: return None
    match = re.search(r"\d+", str(value).replace(",", ""))
    return int(match.group()) if match else None

In [20]:
# ============================================================
# FETCH PLAYER PAGE
# ============================================================

async def get_player_html(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        context = await browser.new_context(
            viewport={"width":1600,"height":1200},
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
        )

        page = await context.new_page()

        response = await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        html = await page.content()

        print(f"HTTP status: {response.status if response else 'None'}")
        print(f"Final URL: {page.url}")
        print(f"HTML length: {len(html):,}")
        print(f"Player page found: {'application/ld+json' in html}")

        await browser.close()

        return html

In [21]:
# ============================================================
# PARSE PLAYER DETAIL + CONTRACTS
# ============================================================

def parse_player_detail(html, url):
    soup = BeautifulSoup(html, "html.parser")

    # ========================================================
    # PLAYER-LEVEL DATA
    # ========================================================

    player = {
        "player":None,
        "player_url":url,

        "leadership_role":None,
        "sweater_number":None,
        "age":None,
        "position":None,
        "shoots_catches":None,
        "height":None,
        "height_inches":None,
        "weight_lbs":None,

        "depth_chart_position":None,
        "depth_chart_line":None,

        "drafted":False,
        "draft_round":None,
        "draft_pick":None,
        "draft_year":None,

        "agent":None,
        "birthdate":None,
        "birthplace":None,
        "nationality":None,

        "ufa_year":None,
        "elc_age":None,
        "waivers_eligibility":None,
        "estimated_career_earnings":None,
    }

    # --------------------------------------------------------
    # JSON-LD
    # --------------------------------------------------------

    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
        except (json.JSONDecodeError, TypeError):
            continue

        if data.get("@type") != "SportsTeam":
            continue

        role = data.get("member", {})
        person = role.get("member", {})

        if person.get("@type") != "Person":
            continue

        player["player"] = person.get("name")
        player["sweater_number"] = parse_int(role.get("numberedPosition"))

        position_map = {
            "Center":"C",
            "Left Wing":"LW",
            "Right Wing":"RW",
            "Defense":"D",
            "Defence":"D",
            "Goalie":"G",
            "Goaltender":"G",
        }

        role_name = role.get("roleName")
        player["position"] = position_map.get(role_name, role_name)

        player["birthdate"] = person.get("birthDate")
        player["nationality"] = person.get("nationality")

        height = person.get("height", {})
        weight = person.get("weight", {})
        earnings = person.get("netWorth", {})

        player["height_inches"] = parse_int(height.get("value"))
        player["weight_lbs"] = parse_int(weight.get("value"))
        player["estimated_career_earnings"] = parse_money(earnings.get("value"))

        if player["height_inches"]:
            feet, inches = divmod(player["height_inches"], 12)
            player["height"] = f"{feet}'{inches}\""

        if player["birthdate"]:
            try:
                dob = datetime.strptime(player["birthdate"], "%Y-%m-%d").date()
                today = datetime.now().date()
                player["age"] = today.year - dob.year - ((today.month,today.day) < (dob.month,dob.day))
            except ValueError:
                pass

        break

    # --------------------------------------------------------
    # PLAYER HEADER
    # --------------------------------------------------------

    for label in soup.select(".pp_subset"):
        key = clean_text(label.get_text(" ", strip=True))
        parent = label.parent
        if not parent:
            continue

        value_element = parent.select_one(".statsrow_val")
        if not value_element:
            continue

        value = clean_text(value_element.get_text(" ", strip=True))
        if not value:
            continue

        key_lower = key.lower() if key else ""

        if key_lower == "#":
            player["sweater_number"] = parse_int(value)

        elif key_lower == "age":
            player["age"] = parse_int(value)

        elif key_lower == "pos":
            player["position"] = value.upper()

        elif key_lower in {"shot","catches"}:
            player["shoots_catches"] = value.upper()

        elif key_lower == "h":
            player["height"] = value

            match = re.search(r"(\d+)['′]\s*(\d+)", value)

            if match:
                player["height_inches"] = int(match.group(1))*12 + int(match.group(2))

        elif key_lower == "w":
            player["weight_lbs"] = parse_int(value)

    # --------------------------------------------------------
    # LEADERSHIP
    # --------------------------------------------------------

    for text in soup.stripped_strings:
        value = clean_text(text)

        if value in {"Captain","A. Captain","Alternate Captain"}:
            player["leadership_role"] = value
            break

    # --------------------------------------------------------
    # DEPTH CHART
    # --------------------------------------------------------

    depth = soup.select_one(".pp_dc")

    if depth:
        chip = depth.select_one(".pp_dc_chip")
        value = depth.select_one(".pp_dc_value")

        if chip:
            chip_text = clean_text(chip.get_text(" ", strip=True))

            if chip_text:
                match = re.match(r"([A-Za-z]+)(\d+)", chip_text)

                if match:
                    player["depth_chart_position"] = match.group(1).upper()
                    player["depth_chart_line"] = int(match.group(2))
                else:
                    player["depth_chart_position"] = chip_text

        if value and player["depth_chart_line"] is None:
            player["depth_chart_line"] = parse_int(value.get_text(" ", strip=True))

    # --------------------------------------------------------
    # MICRO PROFILE FIELDS
    # --------------------------------------------------------

    for micro in soup.select(".micro_row"):
        label_element = micro.select_one(".micro_label")
        value_element = micro.select_one(".micro_value")

        if not label_element or not value_element:
            continue

        label = clean_text(label_element.get_text(" ", strip=True))
        value = clean_text(value_element.get_text(" ", strip=True))

        if not label or not value:
            continue

        label_lower = label.lower()

        if "ufa year" in label_lower:
            player["ufa_year"] = parse_int(value)

        elif "elc age" in label_lower:
            player["elc_age"] = parse_int(value)

        elif "waivers eligibility" in label_lower:
            player["waivers_eligibility"] = value

        elif "career earnings" in label_lower:
            amount = parse_money(value)

            if amount is not None:
                player["estimated_career_earnings"] = amount

    # --------------------------------------------------------
    # DRAFT
    # --------------------------------------------------------

    draft_label = soup.find(
        "span",
        string=lambda x: x and x.strip() == "Draft Team"
    )

    if draft_label:
        player["drafted"] = True
        draft_values = draft_label.parent.find_next_sibling("div")

        if draft_values:
            for item in draft_values.find_all("div", recursive=False):
                text = clean_text(item.get_text(" ", strip=True))

                if not text:
                    continue

                match = re.search(r"Round\s+(\d+)", text, re.I)
                if match:
                    player["draft_round"] = int(match.group(1))
                    continue

                match = re.search(r"Pick\s+(\d+)", text, re.I)
                if match:
                    player["draft_pick"] = int(match.group(1))
                    continue

                match = re.search(r"Year\s+(20\d{2})", text, re.I)
                if match:
                    player["draft_year"] = int(match.group(1))

    # --------------------------------------------------------
    # AGENT / BIRTHPLACE
    # --------------------------------------------------------

    for label_node in soup.find_all(string=re.compile(r"^(Agent|Born|Birthplace)$", re.I)):
        label = clean_text(str(label_node))
        parent = label_node.parent

        if not parent or not parent.parent:
            continue

        text = clean_text(parent.parent.get_text(" ", strip=True))

        if not text:
            continue

        value = re.sub(
            rf"^{re.escape(label)}\s*:?\s*",
            "",
            text,
            flags=re.I
        ).strip()

        if label.lower() == "agent" and value:
            player["agent"] = value

        elif label.lower() in {"born","birthplace"} and value and not player["birthplace"]:
            player["birthplace"] = value

    # ========================================================
    # CONTRACT TABS
    # ========================================================

    rows = []

    tabs = soup.select("button[id^='tab-']")

    for contract_number, tab in enumerate(tabs, 1):
        tab_text = clean_text(tab.get_text(" ", strip=True))

        if not tab_text:
            continue

        # Expected:
        # 2023-2026 $950K x 3
        # 2026-2031 $18.00M x 5

        match = re.search(
            r"(20\d{2})-(20\d{2}).*?\$([\d.]+)\s*([KMB]?).*?x\s*(\d+)",
            tab_text,
            re.I
        )

        if not match:
            continue

        start_year = int(match.group(1))
        end_year = int(match.group(2))
        cap_hit = parse_money(f"{match.group(3)}{match.group(4)}")
        term = int(match.group(5))

        row = dict(player)

        row.update({
            "contract_number":contract_number,
            "current_contract":False,

            "team":None,
            "contract_type":None,

            "season_from":f"{start_year}-{str(start_year+1)[-2:]}",
            "season_to":f"{end_year-1}-{str(end_year)[-2:]}",

            "cap_hit":cap_hit,
            "term":term,
            "total_value":None,

            "signing_status":None,
            "signing_age":None,

            "expiry_status":None,
            "expiry_year":end_year,
            "expiry_age":None,

            "signed_date":None,
            "pct_cap_contract_start":None,
            "signing_gm":None,
            "signing_agent":None,
            "offer_sheet":None,
        })

        # ----------------------------------------------------
        # YEARLY CONTRACT FIELDS
        # ----------------------------------------------------

        for year in range(1, 9):
            row[f"cap_hit_yr{year}"] = None
            row[f"aav_yr{year}"] = None
            row[f"base_salary_yr{year}"] = None
            row[f"performance_bonus_yr{year}"] = None
            row[f"signing_bonus_yr{year}"] = None
            row[f"total_salary_yr{year}"] = None
            row[f"minors_salary_yr{year}"] = None
            row[f"clauses_yr{year}"] = None

        rows.append(row)

    # ========================================================
    # IDENTIFY CURRENT CONTRACT
    # ========================================================

    current_heading = None

    for heading in soup.find_all(["h2","h3","h4"]):
        if "current contract" in heading.get_text(" ", strip=True).lower():
            current_heading = heading
            break

    if current_heading:
        contract_parts = []
        node = current_heading

        for _ in range(120):
            node = node.find_next()

            if node is None:
                break

            if node.name in {"h2","h3"} and node is not current_heading:
                break

            if node.name:
                text = clean_text(node.get_text(" ", strip=True))

                if text:
                    contract_parts.append(text)

        contract_text = " ".join(contract_parts)

        # ----------------------------------------------------
        # CURRENT CAP HIT
        # ----------------------------------------------------

        match = re.search(
            r"Cap Hit\s+\$([\d,.]+(?:\s*[KMB])?)",
            contract_text,
            re.I
        )

        current_cap_hit = parse_money(match.group(1)) if match else None

        # Match current contract to tab
        if current_cap_hit is not None:
            matching_rows = [
                row for row in rows
                if row["cap_hit"] == current_cap_hit
            ]

            if len(matching_rows) == 1:
                matching_rows[0]["current_contract"] = True

        current_row = next(
            (row for row in rows if row["current_contract"]),
            None
        )

        if current_row:

            # ------------------------------------------------
            # EXPIRY
            # ------------------------------------------------

            match = re.search(
                r"Expiry Status\s+(UFA|RFA|10\.2\(c\)|10\.2c)\s+(20\d{2})\s+Age\s+(\d+)",
                contract_text,
                re.I
            )

            if match:
                current_row["expiry_status"] = match.group(1).upper()
                current_row["expiry_year"] = int(match.group(2))
                current_row["expiry_age"] = int(match.group(3))

            # ------------------------------------------------
            # SIGNING STATUS / AGE
            # ------------------------------------------------

            match = re.search(
                r"Signing Status\s+([A-Za-z0-9().]+)\s+Age\s+(\d+)",
                contract_text,
                re.I
            )

            if match:
                current_row["signing_status"] = match.group(1).upper()
                current_row["signing_age"] = int(match.group(2))

            # ------------------------------------------------
            # TOTAL VALUE
            # ------------------------------------------------

            match = re.search(
                r"Total Value\s+\$([\d,.]+(?:\s*[KMB])?)",
                contract_text,
                re.I
            )

            if match:
                current_row["total_value"] = parse_money(match.group(1))

            # ------------------------------------------------
            # SIGNED DATE
            # ------------------------------------------------

            match = re.search(
                r"Signed\s+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
                r"\s+\d{1,2},\s+\d{4}",
                contract_text,
                re.I
            )

            if match:
                full_match = re.search(
                    r"Signed\s+((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
                    r"\s+\d{1,2},\s+\d{4})",
                    contract_text,
                    re.I
                )

                if full_match:
                    try:
                        current_row["signed_date"] = datetime.strptime(
                            full_match.group(1),
                            "%b %d, %Y"
                        ).date()
                    except ValueError:
                        pass

            # ------------------------------------------------
            # % CAP AT CONTRACT START
            # ------------------------------------------------

            match = re.search(
                r"([\d.]+)%\s*(?:of\s+)?Cap",
                contract_text,
                re.I
            )

            if match:
                current_row["pct_cap_contract_start"] = float(match.group(1))

            # ------------------------------------------------
            # SIGNING GM
            # ------------------------------------------------

            match = re.search(
                r"Signing GM\s+(.+?)(?=\s+Signing Agent|\s+Agent|\s+Offer Sheet|$)",
                contract_text,
                re.I
            )

            if match:
                current_row["signing_gm"] = clean_text(match.group(1))

            # ------------------------------------------------
            # SIGNING AGENT
            # ------------------------------------------------

            match = re.search(
                r"Signing Agent\s+(.+?)(?=\s+Signing GM|\s+Offer Sheet|$)",
                contract_text,
                re.I
            )

            if match:
                current_row["signing_agent"] = clean_text(match.group(1))

            # ------------------------------------------------
            # OFFER SHEET
            # ------------------------------------------------

            if re.search(r"Offer Sheet Matched", contract_text, re.I):
                current_row["offer_sheet"] = "Offer Sheet Matched"

            elif re.search(r"Offer Sheet", contract_text, re.I):
                current_row["offer_sheet"] = "Offer Sheet"

    # ========================================================
    # SOURCE FIELDS
    # ========================================================

    scrape_datetime = datetime.now(timezone.utc)

    for row in rows:
        row["source_url"] = url
        row["scrape_datetime"] = scrape_datetime

    return rows

In [22]:
# ============================================================
# PARSE PLAYER DETAIL + CONTRACTS
# ============================================================

def parse_player_detail(html, url):
    soup = BeautifulSoup(html, "html.parser")

    # ========================================================
    # PLAYER-LEVEL DATA
    # ========================================================

    player = {
        "player":None,
        "player_url":url,

        "leadership_role":None,
        "sweater_number":None,
        "age":None,
        "position":None,
        "shoots_catches":None,
        "height":None,
        "height_inches":None,
        "weight_lbs":None,

        "depth_chart_position":None,
        "depth_chart_line":None,

        "drafted":False,
        "draft_round":None,
        "draft_pick":None,
        "draft_year":None,

        "agent":None,
        "birthdate":None,
        "birthplace":None,
        "nationality":None,

        "ufa_year":None,
        "elc_age":None,
        "waivers_eligibility":None,
        "estimated_career_earnings":None,
    }

    # --------------------------------------------------------
    # JSON-LD
    # --------------------------------------------------------

    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
        except (json.JSONDecodeError, TypeError):
            continue

        if data.get("@type") != "SportsTeam":
            continue

        role = data.get("member", {})
        person = role.get("member", {})

        if person.get("@type") != "Person":
            continue

        player["player"] = person.get("name")
        player["sweater_number"] = parse_int(role.get("numberedPosition"))

        position_map = {
            "Center":"C",
            "Left Wing":"LW",
            "Right Wing":"RW",
            "Defense":"D",
            "Defence":"D",
            "Goalie":"G",
            "Goaltender":"G",
        }

        role_name = role.get("roleName")
        player["position"] = position_map.get(role_name, role_name)

        player["birthdate"] = person.get("birthDate")
        player["nationality"] = person.get("nationality")

        height = person.get("height", {})
        weight = person.get("weight", {})
        earnings = person.get("netWorth", {})

        player["height_inches"] = parse_int(height.get("value"))
        player["weight_lbs"] = parse_int(weight.get("value"))
        player["estimated_career_earnings"] = parse_money(earnings.get("value"))

        if player["height_inches"]:
            feet, inches = divmod(player["height_inches"], 12)
            player["height"] = f"{feet}'{inches}\""

        if player["birthdate"]:
            try:
                dob = datetime.strptime(player["birthdate"], "%Y-%m-%d").date()
                today = datetime.now().date()
                player["age"] = today.year - dob.year - ((today.month,today.day) < (dob.month,dob.day))
            except ValueError:
                pass

        break

    # --------------------------------------------------------
    # PLAYER HEADER
    # --------------------------------------------------------

    for label in soup.select(".pp_subset"):
        key = clean_text(label.get_text(" ", strip=True))
        parent = label.parent
        if not parent:
            continue

        value_element = parent.select_one(".statsrow_val")
        if not value_element:
            continue

        value = clean_text(value_element.get_text(" ", strip=True))
        if not value:
            continue

        key_lower = key.lower() if key else ""

        if key_lower == "#":
            player["sweater_number"] = parse_int(value)

        elif key_lower == "age":
            player["age"] = parse_int(value)

        elif key_lower == "pos":
            player["position"] = value.upper()

        elif key_lower in {"shot","catches"}:
            player["shoots_catches"] = value.upper()

        elif key_lower == "h":
            player["height"] = value

            match = re.search(r"(\d+)['′]\s*(\d+)", value)

            if match:
                player["height_inches"] = int(match.group(1))*12 + int(match.group(2))

        elif key_lower == "w":
            player["weight_lbs"] = parse_int(value)

    # --------------------------------------------------------
    # LEADERSHIP
    # --------------------------------------------------------

    for text in soup.stripped_strings:
        value = clean_text(text)

        if value in {"Captain","A. Captain","Alternate Captain"}:
            player["leadership_role"] = value
            break

    # --------------------------------------------------------
    # DEPTH CHART
    # --------------------------------------------------------

    depth = soup.select_one(".pp_dc")

    if depth:
        chip = depth.select_one(".pp_dc_chip")
        value = depth.select_one(".pp_dc_value")

        if chip:
            chip_text = clean_text(chip.get_text(" ", strip=True))

            if chip_text:
                match = re.match(r"([A-Za-z]+)(\d+)", chip_text)

                if match:
                    player["depth_chart_position"] = match.group(1).upper()
                    player["depth_chart_line"] = int(match.group(2))
                else:
                    player["depth_chart_position"] = chip_text

        if value and player["depth_chart_line"] is None:
            player["depth_chart_line"] = parse_int(value.get_text(" ", strip=True))

    # --------------------------------------------------------
    # MICRO PROFILE FIELDS
    # --------------------------------------------------------

    for micro in soup.select(".micro_row"):
        label_element = micro.select_one(".micro_label")
        value_element = micro.select_one(".micro_value")

        if not label_element or not value_element:
            continue

        label = clean_text(label_element.get_text(" ", strip=True))
        value = clean_text(value_element.get_text(" ", strip=True))

        if not label or not value:
            continue

        label_lower = label.lower()

        if "ufa year" in label_lower:
            player["ufa_year"] = parse_int(value)

        elif "elc age" in label_lower:
            player["elc_age"] = parse_int(value)

        elif "waivers eligibility" in label_lower:
            player["waivers_eligibility"] = value

        elif "career earnings" in label_lower:
            amount = parse_money(value)

            if amount is not None:
                player["estimated_career_earnings"] = amount

    # --------------------------------------------------------
    # DRAFT
    # --------------------------------------------------------

    draft_label = soup.find(
        "span",
        string=lambda x: x and x.strip() == "Draft Team"
    )

    if draft_label:
        player["drafted"] = True
        draft_values = draft_label.parent.find_next_sibling("div")

        if draft_values:
            for item in draft_values.find_all("div", recursive=False):
                text = clean_text(item.get_text(" ", strip=True))

                if not text:
                    continue

                match = re.search(r"Round\s+(\d+)", text, re.I)
                if match:
                    player["draft_round"] = int(match.group(1))
                    continue

                match = re.search(r"Pick\s+(\d+)", text, re.I)
                if match:
                    player["draft_pick"] = int(match.group(1))
                    continue

                match = re.search(r"Year\s+(20\d{2})", text, re.I)
                if match:
                    player["draft_year"] = int(match.group(1))

    # --------------------------------------------------------
    # AGENT / BIRTHPLACE
    # --------------------------------------------------------

    for label_node in soup.find_all(string=re.compile(r"^(Agent|Born|Birthplace)$", re.I)):
        label = clean_text(str(label_node))
        parent = label_node.parent

        if not parent or not parent.parent:
            continue

        text = clean_text(parent.parent.get_text(" ", strip=True))

        if not text:
            continue

        value = re.sub(
            rf"^{re.escape(label)}\s*:?\s*",
            "",
            text,
            flags=re.I
        ).strip()

        if label.lower() == "agent" and value:
            player["agent"] = value

        elif label.lower() in {"born","birthplace"} and value and not player["birthplace"]:
            player["birthplace"] = value

    # ========================================================
    # CONTRACTS
    # ========================================================

    rows = []

    contract_root = soup.select_one("#player-contract-tab-panels")

    if not contract_root:
        return rows

    # --------------------------------------------------------
    # READ CONTRACT OPTIONS FROM ALPINE DATA
    # --------------------------------------------------------

    x_data = contract_root.get("x-data", "")

    options_match = re.search(
        r"options:\s*(\[.*?\])\s*,\s*tabSelected:",
        x_data,
        re.S
    )

    selected_match = re.search(
        r"tabSelected:\s*(\d+)",
        x_data
    )

    if not options_match:
        return rows

    options = json.loads(options_match.group(1))
    selected_contract = int(selected_match.group(1)) if selected_match else None

    # ========================================================
    # EACH CONTRACT
    # ========================================================

    for contract_number, option in enumerate(options, 1):

        contract_id = option["contract_id"]
        contract_value = int(option["value"])

        panel = soup.select_one(f"#c_{contract_id}")

        if not panel:
            continue

        panel_text = clean_text(
            panel.get_text(" ", strip=True)
        )

        row = dict(player)

        # ----------------------------------------------------
        # DEFAULT CONTRACT FIELDS
        # ----------------------------------------------------

        row.update({
            "contract_number":contract_number,
            "contract_id":contract_id,

            "current_contract":contract_value == selected_contract,

            "team":None,
            "contract_type":None,

            "season_from":None,
            "season_to":None,

            "cap_hit":None,
            "term":None,
            "total_value":None,

            "signing_status":None,
            "signing_age":None,

            "expiry_status":None,
            "expiry_year":None,
            "expiry_age":None,

            "signed_date":None,
            "pct_cap_contract_start":None,

            "signing_gm":None,
            "signing_agent":None,
            "offer_sheet":None,
        })

        # ----------------------------------------------------
        # TEAM
        # ----------------------------------------------------

        # Player's current team from JSON-LD.
        for script in soup.find_all(
            "script",
            type="application/ld+json"
        ):
            try:
                data = json.loads(script.string or "")
            except:
                continue

            if data.get("@type") == "SportsTeam":
                row["team"] = data.get("name")
                break

        # ----------------------------------------------------
        # CONTRACT SEASONS
        # ----------------------------------------------------

        match = re.search(
            r"\b(20\d{2}-\d{2})\s+to\s+(20\d{2}-\d{2})\b",
            panel_text
        )

        if match:
            row["season_from"] = match.group(1)
            row["season_to"] = match.group(2)

        # ----------------------------------------------------
        # CONTRACT TYPE
        # ----------------------------------------------------

        match = re.search(
            r"\b(Entry Level Contract|Standard Player Contract|Standard Contract)\b",
            panel_text,
            re.I
        )

        if match:
            row["contract_type"] = match.group(1)

        # ----------------------------------------------------
        # CAP HIT
        # ----------------------------------------------------

        match = re.search(
            r"Cap Hit\s+\$([\d,]+)",
            panel_text,
            re.I
        )

        if match:
            row["cap_hit"] = int(
                match.group(1).replace(",", "")
            )

        # ----------------------------------------------------
        # TERM
        # ----------------------------------------------------

        match = re.search(
            r"Term\s+(\d+)\s+years?",
            panel_text,
            re.I
        )

        if match:
            row["term"] = int(match.group(1))

        # ----------------------------------------------------
        # TOTAL VALUE
        # ----------------------------------------------------

        match = re.search(
            r"Total Value\s+\$([\d,]+)",
            panel_text,
            re.I
        )

        if match:
            row["total_value"] = int(
                match.group(1).replace(",", "")
            )

        # ----------------------------------------------------
        # SIGNING STATUS
        # ----------------------------------------------------

        match = re.search(
            r"Signing Status\s+([A-Za-z0-9.()]+)\s+age\s+(\d+)",
            panel_text,
            re.I
        )

        if match:
            row["signing_status"] = match.group(1).upper()
            row["signing_age"] = int(match.group(2))

        # ----------------------------------------------------
        # EXPIRY STATUS
        # ----------------------------------------------------

        match = re.search(
            r"Expiry Status\s+([A-Za-z0-9.()]+)\s+(20\d{2})\s+age\s+(\d+)",
            panel_text,
            re.I
        )

        if match:
            row["expiry_status"] = match.group(1).upper()
            row["expiry_year"] = int(match.group(2))
            row["expiry_age"] = int(match.group(3))

        # ----------------------------------------------------
        # SIGNED DATE
        # ----------------------------------------------------

        match = re.search(
            r"Signed\s+((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
            r"\s+\d{1,2},\s+\d{4})",
            panel_text,
            re.I
        )

        if match:
            try:
                row["signed_date"] = datetime.strptime(
                    match.group(1),
                    "%b %d, %Y"
                ).date()
            except ValueError:
                pass

        # ----------------------------------------------------
        # % CAP CONTRACT START
        # ----------------------------------------------------

        match = re.search(
            r"% Cap Contract Start\s+([\d.]+)%",
            panel_text,
            re.I
        )

        if match:
            row["pct_cap_contract_start"] = float(
                match.group(1)
            )

        # ----------------------------------------------------
        # SIGNING GM
        # ----------------------------------------------------

        label = panel.find(
            string=lambda x:
                x and clean_text(x) == "Signing GM"
        )

        if label:
            parent = label.parent

            if parent and parent.parent:
                spans = parent.parent.find_all("span", recursive=False)

                if len(spans) >= 2:
                    row["signing_gm"] = clean_text(
                        spans[-1].get_text(" ", strip=True)
                    )

        # ----------------------------------------------------
        # SIGNING AGENT
        # ----------------------------------------------------

        label = panel.find(
            string=lambda x:
                x and clean_text(x) == "Signing Agent"
        )

        if label:
            parent = label.parent

            if parent and parent.parent:
                spans = parent.parent.find_all("span", recursive=False)

                if len(spans) >= 2:
                    row["signing_agent"] = clean_text(
                        spans[-1].get_text(" ", strip=True)
                    )

        # ----------------------------------------------------
        # OFFER SHEET
        # ----------------------------------------------------

        if re.search(
            r"Offer Sheet\s+Offer Sheet Matched",
            panel_text,
            re.I
        ):
            row["offer_sheet"] = "Offer Sheet Matched"

        elif re.search(
            r"\bOffer Sheet\b",
            panel_text,
            re.I
        ):
            row["offer_sheet"] = "Offer Sheet"

        # ====================================================
        # ANNUAL CONTRACT FIELDS
        # ====================================================

        for yr in range(1, 9):
            row[f"cap_hit_yr{yr}"] = None
            row[f"aav_yr{yr}"] = None
            row[f"base_salary_yr{yr}"] = None
            row[f"performance_bonus_yr{yr}"] = None
            row[f"signing_bonus_yr{yr}"] = None
            row[f"total_salary_yr{yr}"] = None
            row[f"minors_salary_yr{yr}"] = None
            row[f"clauses_yr{yr}"] = None

        # ----------------------------------------------------
        # CONTRACT SALARY TABLE
        # ----------------------------------------------------

        table = panel.find("table")

        if table:

            annual_field_map = {
                "cap hit":"cap_hit",
                "aav":"aav",
                "base salary":"base_salary",
                "perf. bonus":"performance_bonus",
                "performance bonus":"performance_bonus",
                "signing bonus":"signing_bonus",
                "total salary":"total_salary",
                "minors salary":"minors_salary",
                "clauses":"clauses",
            }

            table_rows = table.find_all("tr")

            # Header:
            # blank | 2026-27 | 2027-28 ... | 2031
            headers = []

            if table_rows:
                header_cells = table_rows[0].find_all(
                    ["th","td"]
                )

                headers = [
                    clean_text(cell.get_text(" ", strip=True))
                    for cell in header_cells
                ]

            # Contract years only.
            # Excludes expiry column such as 2031.
            contract_year_indexes = []

            for idx, header in enumerate(headers):

                if re.fullmatch(
                    r"20\d{2}-\d{2}",
                    header or ""
                ):
                    contract_year_indexes.append(idx)

            for tr in table_rows[1:]:

                cells = tr.find_all("td")

                if not cells:
                    continue

                label = clean_text(
                    cells[0].get_text(" ", strip=True)
                ).lower()

                field = annual_field_map.get(label)

                if not field:
                    continue

                for yr, cell_index in enumerate(
                    contract_year_indexes,
                    1
                ):

                    if yr > 8:
                        break

                    if cell_index >= len(cells):
                        continue

                    cell = cells[cell_index]

                    value_text = clean_text(
                        cell.get_text(" ", strip=True)
                    )

                    # Clauses are text rather than money
                    if field == "clauses":

                        row[f"{field}_yr{yr}"] = (
                            value_text
                            if value_text
                            else None
                        )

                        continue

                    # Prefer the exact long-form money value
                    value_element = cell.select_one(
                        ".val-lg"
                    )

                    if value_element:
                        money_text = clean_text(
                            value_element.get_text(
                                " ",
                                strip=True
                            )
                        )

                        row[f"{field}_yr{yr}"] = (
                            parse_money(money_text)
                        )

                    elif value_text:
                        row[f"{field}_yr{yr}"] = (
                            parse_money(value_text)
                        )

        # ----------------------------------------------------
        # SOURCE
        # ----------------------------------------------------

        row["source_url"] = url
        row["scrape_datetime"] = datetime.now(
            timezone.utc
        )

        rows.append(row)

    return rows

In [23]:
# ============================================================
# RUN
# ============================================================
html = await get_player_html(PLAYER_URL)

rows = parse_player_detail(
    html=html,
    url=PLAYER_URL
)

df_detail = pd.DataFrame(rows)


HTTP status: 200
Final URL: https://puckpedia.com/player/leo-carlsson
HTML length: 356,572
Player page found: True


In [24]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

display(df_detail.T)

,0,1
player,Leo Carlsson,Leo Carlsson
player_url,https://puckpedia.com/player/leo-carlsson,https://puckpedia.com/player/leo-carlsson
leadership_role,A. Captain,A. Captain
sweater_number,91,91
age,21,21
position,C,C
shoots_catches,L,L
height,"6'3""","6'3"""
height_inches,75,75
weight_lbs,207,207


In [19]:
import time
from playwright.async_api import async_playwright

async def speed_test_player(url):
    timings = {}

    total = time.time()

    t = time.time()
    p = await async_playwright().start()
    browser = await p.chromium.launch(headless=True)
    timings["launch"] = time.time() - t

    t = time.time()
    context = await browser.new_context(
        viewport={"width":1600,"height":1200},
        user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
    )
    page = await context.new_page()
    timings["context"] = time.time() - t

    t = time.time()
    response = await page.goto(
        url,
        wait_until="domcontentloaded",
        timeout=60000
    )
    timings["goto"] = time.time() - t

    t = time.time()
    html = await page.content()
    timings["content"] = time.time() - t

    await browser.close()
    await p.stop()

    timings["total"] = time.time() - total

    print(f"Status: {response.status if response else None}")
    print(f"HTML length: {len(html):,}")
    print(f"Leo found: {'Leo Carlsson' in html}")
    print(f"Contract panels: {'player-contract-tab-panels' in html}")
    print(f"Contract 8489: {'c_8489' in html}")
    print(f"Contract 10666: {'c_10666' in html}")

    print("\nTIMINGS")
    for key, value in timings.items():
        print(f"{key:10}: {value:.2f}s")

    return html

html_speed_test = await speed_test_player(
    "https://puckpedia.com/player/leo-carlsson"
)

Status: 200
HTML length: 356,789
Leo found: True
Contract panels: True
Contract 8489: True
Contract 10666: True

TIMINGS
launch    : 0.49s
context   : 0.15s
goto      : 1.35s
content   : 0.04s
total     : 2.07s
